In [1]:
import torch
from torch import nn
from torch.functional import F

In [2]:
model = nn.Sequential(
    nn.Flatten(),
    nn.BatchNorm1d(28*28),
    nn.Linear(28*28, 512),
    nn.ELU(),
    nn.BatchNorm1d(512),
    nn.Linear(512, 256),
    nn.ELU(),
    nn.BatchNorm1d(256),
    nn.Linear(256, 10),
)

In [3]:
! pip install torchsummary

In [4]:
from torchsummary import summary

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# mostra o resumo
summary(model, input_size=(1, 28, 28))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
           Flatten-1                  [-1, 784]               0
       BatchNorm1d-2                  [-1, 784]           1,568
            Linear-3                  [-1, 512]         401,920
               ELU-4                  [-1, 512]               0
       BatchNorm1d-5                  [-1, 512]           1,024
            Linear-6                  [-1, 256]         131,328
               ELU-7                  [-1, 256]               0
       BatchNorm1d-8                  [-1, 256]             512
            Linear-9                   [-1, 10]           2,570
Total params: 538,922
Trainable params: 538,922
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.03
Params size (MB): 2.06
Estimated Total Size (MB): 2.09
-------------------------------------------

In [5]:
print(model[1]) # Accessing the second module (BatchNorm1d)

BatchNorm1d(784, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


You can iterate through the modules:

In [6]:
for i, module in enumerate(model.children()):
  print(f"Module {i}: {module}")

Module 0: Flatten(start_dim=1, end_dim=-1)
Module 1: BatchNorm1d(784, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
Module 2: Linear(in_features=784, out_features=512, bias=True)
Module 3: ELU(alpha=1.0)
Module 4: BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
Module 5: Linear(in_features=512, out_features=256, bias=True)
Module 6: ELU(alpha=1.0)
Module 7: BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
Module 8: Linear(in_features=256, out_features=10, bias=True)


In [7]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Parameter: {name}, Trainable: {param.requires_grad}, Shape: {param.shape}")

Parameter: 1.weight, Trainable: True, Shape: torch.Size([784])
Parameter: 1.bias, Trainable: True, Shape: torch.Size([784])
Parameter: 2.weight, Trainable: True, Shape: torch.Size([512, 784])
Parameter: 2.bias, Trainable: True, Shape: torch.Size([512])
Parameter: 4.weight, Trainable: True, Shape: torch.Size([512])
Parameter: 4.bias, Trainable: True, Shape: torch.Size([512])
Parameter: 5.weight, Trainable: True, Shape: torch.Size([256, 512])
Parameter: 5.bias, Trainable: True, Shape: torch.Size([256])
Parameter: 7.weight, Trainable: True, Shape: torch.Size([256])
Parameter: 7.bias, Trainable: True, Shape: torch.Size([256])
Parameter: 8.weight, Trainable: True, Shape: torch.Size([10, 256])
Parameter: 8.bias, Trainable: True, Shape: torch.Size([10])


In [8]:
model_a = nn.Sequential(
    nn.Flatten(),
    nn.BatchNorm1d(28*28),
    nn.Linear(28*28, 512),
    nn.ELU(),
    nn.BatchNorm1d(512),
    nn.Linear(512, 256),
    nn.ELU(),
    nn.BatchNorm1d(256),
    nn.Linear(256, 10),
)

import copy

model_b_on_a = copy.deepcopy(model_a)

In [9]:
{summary(model_a, input_size=(1, 28, 28))}
{summary(model_b_on_a, input_size=(1, 28, 28))}

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
           Flatten-1                  [-1, 784]               0
       BatchNorm1d-2                  [-1, 784]           1,568
            Linear-3                  [-1, 512]         401,920
               ELU-4                  [-1, 512]               0
       BatchNorm1d-5                  [-1, 512]           1,024
            Linear-6                  [-1, 256]         131,328
               ELU-7                  [-1, 256]               0
       BatchNorm1d-8                  [-1, 256]             512
            Linear-9                   [-1, 10]           2,570
Total params: 538,922
Trainable params: 538,922
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.03
Params size (MB): 2.06
Estimated Total Size (MB): 2.09
-------------------------------------------

{None}

In [10]:
# Convert the generator to a list to enable slicing
parameters_to_freeze = list(model_b_on_a.named_parameters())[:-2]

for name, param in parameters_to_freeze:
  param.requires_grad = False

# Verify which parameters are trainable
for name, param in model_b_on_a.named_parameters():
    print(f"Parameter: {name}, Trainable: {param.requires_grad}")

Parameter: 1.weight, Trainable: False
Parameter: 1.bias, Trainable: False
Parameter: 2.weight, Trainable: False
Parameter: 2.bias, Trainable: False
Parameter: 4.weight, Trainable: False
Parameter: 4.bias, Trainable: False
Parameter: 5.weight, Trainable: False
Parameter: 5.bias, Trainable: False
Parameter: 7.weight, Trainable: False
Parameter: 7.bias, Trainable: False
Parameter: 8.weight, Trainable: True
Parameter: 8.bias, Trainable: True


In [21]:
optimizer_sgd = torch.optim.SGD(model_b_on_a.parameters(), lr=0.01, momentum=0.9, nesterov=True, weight_decay=0.001)
optimizer_rms = torch.optim.RMSprop(model_b_on_a.parameters(), lr=0.01, weight_decay=0.001, momentum=0.9)
optimizer_adam = torch.optim.Adam(model_b_on_a.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-4)

In [22]:
def exponential_decay_fn(epoch):
    return 0.01 * 0.1**(epoch/20)

In [23]:
def exponential_decay(lr0, s):
  def exponential_decay_fn(epoch):
    return lr0 * 0.1**(epoch/s)
  return exponential_decay_fn

In [30]:
exponential_dec = exponential_decay(lr0=0.01, s=20)
lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer_adam, lr_lambda=exponential_dec)

In [32]:
class SimpleCallback:
    def __init__(self):
        pass

    def on_epoch_end(self, epoch, logs=None):
        print(f"End of epoch {epoch + 1}")
        if logs:
            print(f"Logs: {logs}")